# C1 BiLSTM Hyperparameter Tuning

Random search over the BiLSTM C1 classifier (single vs pileup), with proper safeguards against per-trial overfitting and search-level selection bias.

## What this notebook does

1. Loads the balanced single + pileup dataset (clean singles and synthetic pileups)
2. Splits into train (60%) / val (20%) / **held-out test (20%)** — the test set is NOT used during the search
3. Random-searches over architecture and optimization hyperparameters
4. For each trial, records training history, train/val loss + accuracy, and a per-trial overfit flag
5. Ranks trials by a val-only composite score (excluding overfit trials)
6. Plots loss/accuracy curves for the top 3 finalists
7. Re-trains the top 5 from scratch and evaluates them once on the held-out test set

## Hyperparameters explored

- `lstm_units` — units per direction in each BiLSTM layer
- `n_lstm_layers` — number of stacked BiLSTM layers (1 or 2)
- `dense_units` — width of the dense layer after the LSTMs
- `dropout` — dropout in LSTM and dense layers
- `recurrent_dropout` — dropout inside LSTM recurrent connections
- `learning_rate`
- `batch_size`
- `optimizer` — Adam, AdamW, Nadam

## Overfit safeguards

- **Per-trial:** `overfit_gap = train_acc - val_acc`. Trials with `gap > 0.05` are flagged and excluded from the rankings.
- **Selection bias:** the composite score uses only validation metrics. The test set is held out completely until the very end.
- **Honest final evaluation:** the top 5 finalists are re-trained from scratch and evaluated once on the test set.

In [1]:
import itertools
import random
import time

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

print(f"TensorFlow {tf.__version__}")
print(f"GPUs: {tf.config.list_physical_devices('GPU')}")

I0000 00:00:1775860324.295259 1273311 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


I0000 00:00:1775860324.921780 1273311 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow 2.22.0-dev0+selfbuilt
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Load and split data

Singles come from `processed_waveforms.npz`, restricted to `clean_indices` (the singles never used in pileup generation). Pileups come from `pileup_waveforms.npz`. We balance the two classes, Euclidean-normalize, and split 60/20/20.

In [2]:
singles = np.load("processed_waveforms.npz")
X_singles_all = singles["X_voltage"].astype(np.float32)

pileups = np.load("pileup_waveforms.npz")
X_pileup = pileups["pileup_wf"].astype(np.float32)
n_pileup = X_pileup.shape[0]

clean_idx = pileups["clean_indices"]
X_singles = X_singles_all[clean_idx]
n_singles = X_singles.shape[0]

rng = np.random.default_rng(42)
n_balanced = min(n_singles, n_pileup)
single_idx = rng.choice(n_singles, size=n_balanced, replace=False)
pileup_idx = rng.choice(n_pileup, size=n_balanced, replace=False)

X_all = np.concatenate([X_singles[single_idx], X_pileup[pileup_idx]], axis=0)
y_all = np.concatenate([
    np.zeros(n_balanced, dtype=np.int64),  # 0 = single
    np.ones(n_balanced, dtype=np.int64),   # 1 = pileup
])

# Euclidean-normalize each waveform
norms = np.linalg.norm(X_all, axis=1, keepdims=True)
norms[norms == 0] = 1.0
X_all = X_all / norms

# Shuffle
shuffle = rng.permutation(len(y_all))
X_all = X_all[shuffle]
y_all = y_all[shuffle]

# 60/20/20 train/val/test split
X_tv, X_test, y_tv, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.25, random_state=42, stratify=y_tv
)

# Keras LSTM expects (batch, timesteps, features)
X_train = X_train[..., np.newaxis]
X_val   = X_val[..., np.newaxis]
X_test  = X_test[..., np.newaxis]

print(f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")
print(f"Class balance (train): single={(y_train==0).sum():,}  pileup={(y_train==1).sum():,}")

Train: (208002, 104, 1)  Val: (69335, 104, 1)  Test: (69335, 104, 1)
Class balance (train): single=104,001  pileup=104,001


## Search space

In [3]:
SEARCH_SPACE = {
    "lstm_units":        [32, 64, 128],
    "n_lstm_layers":     [1, 2],
    "dense_units":       [16, 32, 64],
    "dropout":           [0.1, 0.2, 0.3, 0.4],
    "recurrent_dropout": [0.0, 0.1, 0.2],
    "learning_rate":     [1e-4, 5e-4, 1e-3, 2e-3],
    "batch_size":        [256, 512, 1024],
    "optimizer":         ["adam", "adamw", "nadam"],
}

N_TRIALS   = 50
MAX_EPOCHS = 25
PATIENCE   = 6

# Maximum train_acc - val_acc before flagging a trial as overfit
OVERFIT_GAP_THRESHOLD = 0.05

total_combos = 1
for v in SEARCH_SPACE.values():
    total_combos *= len(v)
print(f"Total possible combos: {total_combos:,}")
print(f"Sampling {N_TRIALS} random configurations")

Total possible combos: 7,776
Sampling 50 random configurations


## Model builder

In [4]:
def get_optimizer(name, learning_rate):
    if name == "adam":  return keras.optimizers.Adam(learning_rate=learning_rate)
    if name == "adamw": return keras.optimizers.AdamW(learning_rate=learning_rate)
    if name == "nadam": return keras.optimizers.Nadam(learning_rate=learning_rate)
    raise ValueError(f"Unknown optimizer: {name}")


def build_bilstm(hp, n_samples=104):
    model_layers = [keras.layers.Input(shape=(n_samples, 1))]
    for i in range(hp["n_lstm_layers"]):
        return_seq = (i < hp["n_lstm_layers"] - 1)
        model_layers.append(
            layers.Bidirectional(
                layers.LSTM(
                    hp["lstm_units"],
                    return_sequences=return_seq,
                    dropout=hp["dropout"],
                    recurrent_dropout=hp["recurrent_dropout"],
                )
            )
        )
    model_layers.append(layers.Dense(hp["dense_units"], activation="relu"))
    model_layers.append(layers.Dropout(hp["dropout"]))
    model_layers.append(layers.Dense(1, activation="sigmoid"))

    model = keras.Sequential(model_layers)
    model.compile(
        optimizer=get_optimizer(hp["optimizer"], hp["learning_rate"]),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model

## Trial evaluation

Each trial trains a model on the train split, with early stopping monitored on `val_loss`. After training:

1. Re-measure train and val accuracy/loss
2. Compute `overfit_gap = train_acc - val_acc`
3. Compute `composite_score = 0.5 * val_loss + 0.5 * (1 - val_acc)` (val-only)
4. Save the full training history so we can plot loss curves later

The test set is NOT touched at this stage.

In [5]:
def composite_score(val_loss, val_acc, alpha=0.5):
    return alpha * val_loss + (1 - alpha) * (1 - val_acc)


def run_trial(hp, X_train, X_val, y_train, y_val,
              max_epochs=MAX_EPOCHS, patience=PATIENCE):
    keras.backend.clear_session()
    model = build_bilstm(hp, n_samples=X_train.shape[1])
    n_params = model.count_params()

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=patience,
            restore_best_weights=True, verbose=0,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", patience=3, factor=0.5, verbose=0,
        ),
    ]

    t0 = time.time()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=max_epochs,
        batch_size=hp["batch_size"],
        callbacks=callbacks,
        verbose=0,
    )
    train_time = time.time() - t0

    # Re-measure on train + val after early stopping has restored best weights
    train_loss, train_acc = model.evaluate(X_train, y_train, batch_size=hp["batch_size"], verbose=0)
    val_loss,   val_acc   = model.evaluate(X_val,   y_val,   batch_size=hp["batch_size"], verbose=0)

    overfit_gap = train_acc - val_acc
    is_overfit = overfit_gap > OVERFIT_GAP_THRESHOLD
    best_epoch = int(np.argmin(history.history["val_loss"]) + 1)

    return {
        **hp,
        "n_params": n_params,
        "best_epoch": best_epoch,
        "train_loss": float(train_loss),
        "train_acc": float(train_acc),
        "val_loss": float(val_loss),
        "val_acc": float(val_acc),
        "overfit_gap": float(overfit_gap),
        "is_overfit": bool(is_overfit),
        "composite_score": float(composite_score(val_loss, val_acc)),
        "train_time_s": round(train_time, 1),
    }, history.history

## Run the random search

Histories are kept in memory (a list of dicts) so we can plot loss curves for the top 3 later. The summary results are saved incrementally to `c1_tune_results.csv`.

In [6]:
random.seed(42)
all_combos = list(itertools.product(*SEARCH_SPACE.values()))
keys = list(SEARCH_SPACE.keys())
sampled = random.sample(all_combos, min(N_TRIALS, len(all_combos)))

results = []
histories = []  # parallel list, same order as results

for trial_num, combo in enumerate(sampled):
    hp = dict(zip(keys, combo))
    print(f"\n{'='*78}")
    print(f"Trial {trial_num+1}/{len(sampled)}")
    print(f"  arch: lstm={hp['lstm_units']} layers={hp['n_lstm_layers']} dense={hp['dense_units']} "
          f"drop={hp['dropout']} rdrop={hp['recurrent_dropout']}")
    print(f"  opt:  {hp['optimizer']} lr={hp['learning_rate']} bs={hp['batch_size']}")

    try:
        result, history = run_trial(hp, X_train, X_val, y_train, y_val)
        results.append(result)
        histories.append(history)
        flag = " [OVERFIT]" if result["is_overfit"] else ""
        print(f"  -> train_acc={result['train_acc']:.4f}  val_acc={result['val_acc']:.4f}  "
              f"gap={result['overfit_gap']:+.4f}{flag}  "
              f"score={result['composite_score']:.5f}  "
              f"params={result['n_params']:,}  time={result['train_time_s']}s")
    except Exception as e:
        print(f"  -> FAILED: {e}")
        continue

    # Incremental save (summary only — histories live in memory)
    pd.DataFrame(results).to_csv("c1_tune_results.csv", index=False)

n_overfit = sum(1 for r in results if r['is_overfit'])
print(f"\n{'='*78}")
print(f"Complete: {len(results)}/{len(sampled)} successful, {n_overfit} flagged as overfit")


Trial 1/50
  arch: lstm=128 layers=1 dense=16 drop=0.1 rdrop=0.1
  opt:  adam lr=0.001 bs=256


I0000 00:00:1775860326.890146 1273311 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 12969 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5070 Ti, pci bus id: 0000:01:00.0, compute capability: 12.0a


  -> train_acc=0.9658  val_acc=0.9655  gap=+0.0003  score=0.06636  params=137,249  time=2086.7s

Trial 2/50
  arch: lstm=32 layers=1 dense=64 drop=0.1 rdrop=0.1
  opt:  adam lr=0.0005 bs=512


  -> train_acc=0.9509  val_acc=0.9506  gap=+0.0003  score=0.09212  params=12,929  time=1078.4s

Trial 3/50
  arch: lstm=32 layers=1 dense=16 drop=0.2 rdrop=0.2
  opt:  adam lr=0.001 bs=1024


  -> train_acc=0.8948  val_acc=0.8944  gap=+0.0004  score=0.18700  params=9,761  time=153.3s

Trial 4/50
  arch: lstm=128 layers=1 dense=64 drop=0.1 rdrop=0.0
  opt:  nadam lr=0.001 bs=1024


...

  -> train_acc=0.9364  val_acc=0.9358  gap=+0.0005  score=0.11686  params=37,761  time=36.6s

Trial 47/50
  arch: lstm=32 layers=1 dense=64 drop=0.4 rdrop=0.2
  opt:  adamw lr=0.0005 bs=512


  -> train_acc=0.7312  val_acc=0.7324  gap=-0.0012  score=0.36022  params=12,929  time=301.6s

Trial 48/50
  arch: lstm=32 layers=2 dense=32 drop=0.1 rdrop=0.0
  opt:  nadam lr=0.002 bs=1024


  -> train_acc=0.9634  val_acc=0.9631  gap=+0.0003  score=0.07049  params=35,649  time=83.0s

Trial 49/50
  arch: lstm=128 layers=1 dense=64 drop=0.2 rdrop=0.2
  opt:  nadam lr=0.001 bs=1024


  -> train_acc=0.9567  val_acc=0.9565  gap=+0.0002  score=0.08127  params=149,633  time=531.6s

Trial 50/50
  arch: lstm=64 layers=1 dense=16 drop=0.2 rdrop=0.1
  opt:  adam lr=0.001 bs=512


  -> train_acc=0.9457  val_acc=0.9447  gap=+0.0010  score=0.10098  params=35,873  time=912.1s

Complete: 50/50 successful, 0 flagged as overfit


## Top configurations (overfit excluded)

In [7]:
df = pd.DataFrame(results)
print(f"Total trials: {len(df)}")
print(f"Overfit trials excluded: {df['is_overfit'].sum()}")
print(f"Eligible trials: {(~df['is_overfit']).sum()}")
print()

clean_df = df[~df["is_overfit"]].copy()

print("=" * 100)
print("TOP 10 BY COMPOSITE SCORE (overfit excluded)")
print("=" * 100)
cols = ["lstm_units", "n_lstm_layers", "dense_units", "dropout",
        "recurrent_dropout", "learning_rate", "batch_size", "optimizer",
        "train_acc", "val_acc", "overfit_gap", "composite_score", "n_params", "best_epoch"]
print(clean_df.sort_values("composite_score").head(10)[cols].to_string())

print("\n\nFull comparison: clean vs overfit")
print("=" * 60)
if df['is_overfit'].any():
    summary = df.groupby("is_overfit").agg(
        n=("composite_score", "count"),
        mean_train_acc=("train_acc", "mean"),
        mean_val_acc=("val_acc", "mean"),
        mean_overfit_gap=("overfit_gap", "mean"),
        best_score=("composite_score", "min"),
    )
    print(summary.round(4).to_string())

Total trials: 50
Overfit trials excluded: 0
Eligible trials: 50

TOP 10 BY COMPOSITE SCORE (overfit excluded)
    lstm_units  n_lstm_layers  dense_units  dropout  recurrent_dropout  learning_rate  batch_size optimizer  train_acc   val_acc  overfit_gap  composite_score  n_params  best_epoch
20          32              2           32      0.1                0.1         0.0020         256      adam   0.974827  0.974904    -0.000077         0.050700     35649          16
6           32              2           32      0.1                0.2         0.0020         256     adamw   0.974015  0.973852     0.000163         0.052763     35649          22
0          128              1           16      0.1                0.1         0.0010         256      adam   0.965803  0.965486     0.000317         0.066358    137249          22
4           32              2           64      0.1                0.2         0.0010         512      adam   0.965231  0.964607     0.000624         0.067927     377

## Per-hyperparameter effects

How each hyperparameter individually affects validation accuracy. Red dots = mean for that value.

In [ ]:
import os; os.makedirs("figures", exist_ok=True)
hp_to_plot = list(SEARCH_SPACE.keys())
ncols = 4
nrows = int(np.ceil(len(hp_to_plot) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows))
axes = axes.flatten()

for ax, hp_name in zip(axes, hp_to_plot):
    vals = clean_df[hp_name].astype(str)
    unique = sorted(vals.unique())
    for v in unique:
        mask = vals == v
        ax.scatter([v] * mask.sum(),
                   clean_df.loc[mask, "val_acc"].values * 100,
                   alpha=0.5, s=30, color="tab:blue")
    means = clean_df.groupby(vals)["val_acc"].mean() * 100
    ax.scatter(means.index, means.values, color="tab:red", s=80, zorder=5, label="Mean")
    ax.set_title(hp_name, fontsize=10)
    ax.set_ylabel("Val accuracy [%]", fontsize=8)
    ax.tick_params(axis="x", rotation=30, labelsize=8)
    ax.grid(True, alpha=0.3)

for j in range(len(hp_to_plot), len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Per-hyperparameter effect on validation accuracy (overfit excluded)", fontsize=13)
plt.tight_layout()
plt.savefig("figures/c1_tune_hyperparameter_effects.png", dpi=120, bbox_inches="tight")
plt.show()

## Training curves for the top 3 finalists

These are the configurations that placed best on the val-only composite score (after excluding overfit trials). Each row shows loss and accuracy over epochs for one finalist. Big train/val gaps would indicate residual overfitting that the gap check missed.

In [ ]:
# Map original index in `results` to the clean rank
clean_df = df[~df["is_overfit"]].copy()
top3 = clean_df.sort_values("composite_score").head(3)

fig, axes = plt.subplots(3, 2, figsize=(12, 10))

for plot_row, (orig_idx, row) in enumerate(top3.iterrows()):
    history = histories[orig_idx]

    title = (f"#{plot_row+1}  lstm={int(row.lstm_units)} layers={int(row.n_lstm_layers)} "
             f"dense={int(row.dense_units)} drop={row.dropout} "
             f"lr={row.learning_rate} bs={int(row.batch_size)} {row.optimizer}\n"
             f"val_acc={row.val_acc:.4f}  gap={row.overfit_gap:+.4f}  score={row.composite_score:.5f}")

    # Loss curves
    ax = axes[plot_row, 0]
    epochs = np.arange(1, len(history["loss"]) + 1)
    ax.plot(epochs, history["loss"],     label="train", color="tab:blue")
    ax.plot(epochs, history["val_loss"], label="val",   color="tab:orange")
    ax.axvline(row.best_epoch, color="green", linestyle="--", alpha=0.5, label=f"best epoch ({int(row.best_epoch)})")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss (binary crossentropy)")
    ax.set_title(title, fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # Accuracy curves
    ax = axes[plot_row, 1]
    ax.plot(epochs, np.array(history["accuracy"])     * 100, label="train", color="tab:blue")
    ax.plot(epochs, np.array(history["val_accuracy"]) * 100, label="val",   color="tab:orange")
    ax.axvline(row.best_epoch, color="green", linestyle="--", alpha=0.5)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy [%]")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle("Top 3 finalists — training curves", fontsize=13)
plt.tight_layout()
plt.savefig("figures/c1_tune_top3_training_curves.png", dpi=120, bbox_inches="tight")
plt.show()

## Honest test-set evaluation of top 5 finalists

Re-train each of the top 5 configurations from scratch and evaluate them once on the held-out test set. The `val → test gap` column shows how much the val-based composite score over- or under-estimated the true performance — a large positive gap is a sign of selection bias.

In [10]:
top5 = clean_df.sort_values("composite_score").head(5)
final_results = []

for rank, (_, row) in enumerate(top5.iterrows(), 1):
    hp = {k: row[k] for k in SEARCH_SPACE.keys()}
    # Cast back to native types
    hp["lstm_units"]    = int(hp["lstm_units"])
    hp["n_lstm_layers"] = int(hp["n_lstm_layers"])
    hp["dense_units"]   = int(hp["dense_units"])
    hp["batch_size"]    = int(hp["batch_size"])
    hp["dropout"]       = float(hp["dropout"])
    hp["recurrent_dropout"] = float(hp["recurrent_dropout"])
    hp["learning_rate"] = float(hp["learning_rate"])

    print(f"\nFinalist {rank}: lstm={hp['lstm_units']} layers={hp['n_lstm_layers']} "
          f"dense={hp['dense_units']} drop={hp['dropout']} lr={hp['learning_rate']} "
          f"bs={hp['batch_size']} {hp['optimizer']}")
    print(f"  val_acc (search) = {row.val_acc:.4f}")

    keras.backend.clear_session()
    model = build_bilstm(hp, n_samples=X_train.shape[1])
    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE,
                                       restore_best_weights=True, verbose=0),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=3, factor=0.5, verbose=0),
    ]
    model.fit(X_train, y_train, validation_data=(X_val, y_val),
              epochs=MAX_EPOCHS, batch_size=hp["batch_size"],
              callbacks=callbacks, verbose=0)

    test_loss, test_acc = model.evaluate(X_test, y_test, batch_size=hp["batch_size"], verbose=0)
    val_test_gap = test_acc - row.val_acc

    print(f"  test_acc         = {test_acc:.4f}")
    print(f"  val -> test gap   = {val_test_gap:+.4f}")

    final_results.append({
        "rank": rank,
        "lstm_units": hp["lstm_units"],
        "n_lstm_layers": hp["n_lstm_layers"],
        "dense_units": hp["dense_units"],
        "dropout": hp["dropout"],
        "recurrent_dropout": hp["recurrent_dropout"],
        "learning_rate": hp["learning_rate"],
        "batch_size": hp["batch_size"],
        "optimizer": hp["optimizer"],
        "val_acc": row.val_acc,
        "test_acc": float(test_acc),
        "val_test_gap": float(val_test_gap),
    })

print("\n" + "=" * 70)
print("FINALIST SUMMARY")
print("=" * 70)
final_df = pd.DataFrame(final_results)
print(final_df.to_string(index=False))

df.to_csv("c1_tune_results.csv", index=False)
final_df.to_csv("c1_tune_finalists.csv", index=False)
print("\nSaved c1_tune_results.csv and c1_tune_finalists.csv")


Finalist 1: lstm=32 layers=2 dense=32 drop=0.1 lr=0.002 bs=256 adam
  val_acc (search) = 0.9749


  test_acc         = 0.9740
  val -> test gap   = -0.0009

Finalist 2: lstm=32 layers=2 dense=32 drop=0.1 lr=0.002 bs=256 adamw
  val_acc (search) = 0.9739


  test_acc         = 0.9717
  val -> test gap   = -0.0022

Finalist 3: lstm=128 layers=1 dense=16 drop=0.1 lr=0.001 bs=256 adam
  val_acc (search) = 0.9655


  test_acc         = 0.9669
  val -> test gap   = +0.0015

Finalist 4: lstm=32 layers=2 dense=64 drop=0.1 lr=0.001 bs=512 adam
  val_acc (search) = 0.9646


  test_acc         = 0.9610
  val -> test gap   = -0.0036

Finalist 5: lstm=64 layers=2 dense=64 drop=0.1 lr=0.0005 bs=512 adamw
  val_acc (search) = 0.9641


  test_acc         = 0.9630
  val -> test gap   = -0.0011

FINALIST SUMMARY
 rank  lstm_units  n_lstm_layers  dense_units  dropout  recurrent_dropout  learning_rate  batch_size optimizer  val_acc  test_acc  val_test_gap
    1          32              2           32      0.1                0.1         0.0020         256      adam 0.974904  0.973996     -0.000909
    2          32              2           32      0.1                0.2         0.0020         256     adamw 0.973852  0.971674     -0.002178
    3         128              1           16      0.1                0.1         0.0010         256      adam 0.965486  0.966943      0.001457
    4          32              2           64      0.1                0.2         0.0010         512      adam 0.964607  0.960972     -0.003635
    5          64              2           64      0.1                0.2         0.0005         512     adamw 0.964145  0.963035     -0.001110

Saved c1_tune_results.csv and c1_tune_finalists.csv
